## Part 1 RDDs
Repeat the steps of Assignment 1, i.e. calculation of chi-square values and output of the sorted top terms per category, as well as the joined dictionary, using RDDs and transformations. Write the output to a file output_rdd.txt. Compare the generated output_rdd.txt with your generated output.txt from Assignment 1 and describe your observations briefly in the submission report (see Part 3).

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

from pyspark.sql.functions import explode, split, lower


import json
import re
from datetime import datetime
# Stop existing SparkContext if it's running
if SparkContext._active_spark_context:
    SparkContext._active_spark_context.stop()



In [2]:
# sc = SparkContext(appName="CheckParallelism")
# print("Default parallelism:", sc.defaultParallelism)

In [3]:
# Initialize Spark context and session
conf = SparkConf().setAppName("ChiSquareAnalysis")
#sc.stop()
sc = SparkContext(conf=conf)
spark = SparkSession(sc)

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/lib/spark/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/lib/hadoop/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/05/13 11:59:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/05/13 11:59:29 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/05/13 11:59:29 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/05/13 11:59:29 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
25/05/13 11:59:29 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
25/05/13 11:59:29 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
25/05/13 11:59:29 WARN Utils: Service 'SparkUI' could not bind on port 4046. Attempting port 4047.
25/05/13 11:59:29 WARN Utils: Service 'SparkUI' could not bind on port 4047. Attempting port 4048.
25/05/13 11:59:29 WARN Utils: Service 'SparkUI' could not bind on port 4048. Attempting port 4049.
25/05/13 11:59:29 WARN Utils: Service 'SparkUI' could not bind on port 4049. Attempting port 4050.
25/05/13 1

In [4]:
spark

### Preprocessing and Intermediate Step

In [5]:
# Define the stopwords file 
stopwords_file = "stopwords.txt"

# Load stopwords into a set
with open(stopwords_file, "r") as f:
    stopwords = set(f.read().strip().split())
    
# Load and preprocess the Amazon reviews dataset
#input_file = "hdfs:///user/dic25_shared/amazon-reviews/full/reviews_devset.json"
input_file = "hdfs:///user/dic25_shared/amazon-reviews/full/reviewscombined.json"
reviews_rdd = sc.textFile(input_file)

In [6]:
# Show first two objects of reviews_rdd
reviews_rdd.take(2)

['{"reviewerID": "A2VNYWOPJ13AFP", "asin": "0981850006", "reviewerName": "Amazon Customer \\"carringt0n\\"", "helpful": [6, 7], "reviewText": "This was a gift for my other husband.  He\'s making us things from it all the time and we love the food.  Directions are simple, easy to read and interpret, and fun to make.  We all love different kinds of cuisine and Raichlen provides recipes from everywhere along the barbecue trail as he calls it. Get it and just open a page.  Have at it.  You\'ll love the food and it has provided us with an insight into the culture that produced it. It\'s all about broadening horizons.  Yum!!", "overall": 5.0, "summary": "Delish", "unixReviewTime": 1259798400, "reviewTime": "12 3, 2009", "category": "Patio_Lawn_and_Garde"}',
 '{"reviewerID": "A20DWVV8HML3AW", "asin": "0981850006", "reviewerName": "Cyndy", "helpful": [0, 0], "reviewText": "My husband rarely asks for anything specific, but he really liked this show.  I was so glad I could find this for him.  He

In [7]:
def preprocess_text(text):
    text = text.lower()
    unigrams = re.split(r'\s+|\d+|[(){}[\].!?,;:+=_"\'`~#@&*%€$§\\/\-]', text)
    unigrams = set(unigrams)
    return unigrams

def valid_word(word):
    if len(word) > 1 and word not in stopwords:
        return word
    
json_rdd = reviews_rdd.map(lambda line: json.loads(line))

word_category_rdd = json_rdd.flatMap(lambda x: [(x["category"], word) for word in preprocess_text(x['reviewText']) if valid_word(word)])

In [8]:
word_category_rdd.take(1)

[('Patio_Lawn_and_Garde', 'insight')]

In [9]:
# parse JSON and extract (category, word) pairs
word_cat_rdd = (
    reviews_rdd
      .map(json.loads)
      .flatMap(lambda r: [
          ((r["category"], w), 1)
          for w in preprocess_text(r.get("reviewText", ""))
          if valid_word(w)          
      ])
)
# quickly peek at a few tokens (no shuffle)
print("Sample tokens:", word_cat_rdd.take(5))

# approximate counts on a 1% sample
sample_counts = (
    word_cat_rdd
    .sample(False, 0.01, seed=42)
    .reduceByKey(lambda a, b: a + b)
)
print("Approximate top-10 on 1% sample:", sample_counts.takeOrdered(10, key=lambda kv: -kv[1]))

# full counts with map‑side combine for fewer network shuffles
combined_counts = word_cat_rdd.combineByKey(
    createCombiner=lambda v: v,
    mergeValue=lambda acc, v: acc + v,
    mergeCombiners=lambda acc1, acc2: acc1 + acc2,
    numPartitions=sc.defaultParallelism * 2             
)




Sample tokens: [(('Patio_Lawn_and_Garde', 'insight'), 1), (('Patio_Lawn_and_Garde', 'things'), 1), (('Patio_Lawn_and_Garde', 'raichlen'), 1), (('Patio_Lawn_and_Garde', 'open'), 1), (('Patio_Lawn_and_Garde', 'horizons'), 1)]


Approximate top-10 on 1% sample: [(('Book', 'great'), 48752), (('Book', 'good'), 48105), (('Book', 'reading'), 39074), (('Book', 'love'), 37926), (('Book', 'time'), 37358), (('Book', 'author'), 32388), (('Book', 'characters'), 31265), (('Book', 'written'), 25334), (('Book', 'recommend'), 24450), (('Book', 'series'), 23772)]


In [10]:
reduced_rdd = combined_counts.map(
    lambda x: (x[0][0], (x[0][1], x[1]))
).cache()

In [11]:
reviews_per_category_count_rdd = (
    json_rdd
    .map(lambda r: (r['category'], 1))
    .reduceByKey(lambda a, b: a + b)
).cache()

### Calculate Chi-Square

Next, we prepare the values A, B, C and D which are needed to compute the chi-square.

In [12]:
start_time = datetime.now()

# A: docs in c containing t
A_value_rdd = reduced_rdd.map(
    lambda x: ((x[0], x[1][0]), x[1][1])
)

# B_total: total docs containing t across all categories
B_value_complete_rdd = (
    reduced_rdd
    .map(lambda x: (x[1][0], x[1][1]))
    .reduceByKey(lambda a, b: a + b)
)

# B: docs not in c containing t = B_total – A
A_B_value_rdd = (
    A_value_rdd
    .map(lambda x: (x[0][1], (x[0][0], x[1])))
    .join(B_value_complete_rdd)
    .map(lambda x: (
        (x[1][0][0], x[0]),
        (x[1][0][1], x[1][1] - x[1][0][1])
    ))
)

# C: docs in c without t = docs_in_c – A
C_value_rdd = (
    reduced_rdd
    .join(reviews_per_category_count_rdd)
    .map(lambda x: (
        (x[0], x[1][0][0]),
        x[1][1] - x[1][0][1]
    ))
)

# Now get the global doc count to compute D
dataset_length = reviews_rdd.count()

# A, B, C, D combined
A_B_C_D_values_rdd = (
    A_B_value_rdd
    .join(C_value_rdd)
    .map(lambda x: (
        x[0],
        (
            x[1][0][0],  # A
            x[1][0][1],  # B
            x[1][1],     # C
            dataset_length - (
                x[1][0][0] + x[1][0][1] + x[1][1]
            )
        )
    ))
)

Start time: 2025-05-13 12:59:22.980910
End time:   2025-05-13 14:46:56.136444
Elapsed:    1:47:33.155534


In [ ]:
# chi-square and top features
chi_square_rdd = A_B_C_D_values_rdd.map(lambda x: (
    x[0][0],  
    (
        x[0][1],  
        (
            (dataset_length * 
             (x[1][0] * x[1][3] - x[1][1] * x[1][2]) ** 2) /
            ((x[1][0] + x[1][1]) *
             (x[1][0] + x[1][2]) *
             (x[1][1] + x[1][3]) *
             (x[1][2] + x[1][3]))
        )
    )
))

# Sort within each category and take top 75 words
chi_sort_rdd = chi_square_rdd.sortBy(
    lambda x: (x[0], x[1][1]), ascending=False
)

chi_cropped_rdd = (
    chi_sort_rdd
    .groupByKey()
    .map(lambda x: (x[0], list(x[1])[:75]))
    .sortByKey()
)


In [13]:
# Get the output and stop the time
data = chi_cropped_rdd.collect()

# flatten to unique words
words = sorted({w for _, wl in data for (w, _) in wl})

end_time = datetime.now()

with open("output_rdd2.txt", "w") as writer:
    for row in data:
        writer.write(str(row) + "\n")
    writer.write("\n# Unique top-75 words across all categories:\n")
    writer.write(" ".join(words))

print(f"Start time: {start_time}")
print(f"End time:   {end_time}")
print(f"Elapsed:    {end_time - start_time}")

In [ ]:
spark.stop()